<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: Trend, season and noise

**[Trend, season and noise](https://learning.nextia-ai.com/courses/forecasting/m02/trend-season-and-noise/)** · Forecasting and Time Series: Predict What Comes Next · Module 2, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** split a time series into a slow trend, seasons that repeat and the rest, with MSTL, and to read each part.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The Larkfield tables, from [Meet the Larkfield data](https://learning.nextia-ai.com/courses/forecasting/m01/meet-the-larkfield-data/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m02/trend-season-and-noise/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M02-L01-trend-season-and-noise/trend-season-and-noise-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A tiny series: three parts added up

A shop sells 100 units on day 1, and 2 more every day after that: that is the **trend**, the slow change. On Saturdays it sells 30 more and on Sundays 20 more: that is the **season**, a pattern that repeats every week. Every day also has a small surprise: the **remainder** (or noise).

> **Predict.** On day 13 (a Saturday), the trend is 124. The remainder is −3. What is the real value? Then run the cell.

In [ ]:
days = np.arange(1, 15)
trend = 100 + 2 * (days - 1)
season = np.tile([0, 0, 0, 0, 0, 30, 20], 2)          # Monday ... Sunday, two weeks
remainder = np.array([1, -2, 0, 3, -1, 2, -4, 0, 2, -1, 1, -2, -3, 2])
value = trend + season + remainder
print(pd.DataFrame({"day": days, "trend": trend, "season": season, "remainder": remainder, "value": value}).tail(3).to_string(index=False))

> **Check.** You should see day 13 with trend 124, season 30, remainder −3 and value 151.

A **decomposition** does the opposite: it starts from the values and estimates the three parts. The methods STL and MSTL do this with local averages. MSTL can take **several seasons** at once: here a week (7 days) and a year (365 days).

## 2. Larkfield's daily orders

The next cell adds up the three order channels per day, fills the day that the feed dropped (2025-10-19) with the mean of the same weekday one week before and after, and draws the series.

In [ ]:
import matplotlib.pyplot as plt

orders = pd.read_csv("data/orders.csv", parse_dates=["date"]).groupby("date")["orders"].sum().asfreq("D")
before, after = orders.shift(7), orders.shift(-7)
orders = orders.fillna((before + after) / 2).round()
print(len(orders), "days, first", int(orders.iloc[0]), "last", int(orders.iloc[-1]), "max", int(orders.max()), "on", orders.idxmax().date())
orders.plot(figsize=(11, 3.5), lw=0.8, xlabel="Day", ylabel="Orders per day")
plt.show()

> **Check.** You should see `728 days`, and a maximum on `2025-07-05`: the Saturday of the heat wave, a situation we planted. The chart rises every spring, falls in the winter, and has a tall spike every November (Black Friday).

## 3. MSTL: trend, two seasons and the rest

statsforecast's `MSTL` takes the seasons as a list of lengths. The fitted model keeps the parts in a table with the columns `trend`, `seasonal7`, `seasonal365` and `remainder`.

> **Predict.** Which part changes the orders the most over the two years: the trend, the weekly season or the yearly season? Then run the cell.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import MSTL

df = pd.DataFrame({"unique_id": "orders", "ds": orders.index, "y": orders.to_numpy()})
sf = StatsForecast(models=[MSTL(season_length=[7, 365])], freq="D")
sf.fit(df)
parts = sf.fitted_[0, 0].model_
parts.index = orders.index
share = {c: round(float(100 * parts[c].var() / parts["data"].var()), 1) for c in ["trend", "seasonal7", "seasonal365", "remainder"]}
print("share of the variance, %:", share)
print("trend: first", round(parts["trend"].iloc[0]), "last", round(parts["trend"].iloc[-1]))
parts[["data", "trend", "seasonal7", "seasonal365", "remainder"]].plot(subplots=True, figsize=(11, 8), lw=0.7, legend=True)
plt.show()

> **Check.** You should see `{'trend': 6.7, 'seasonal7': 6.3, 'seasonal365': 87.2, 'remainder': 16.8}` and `trend: first 612 last 759`.

The **yearly season** is by far the largest part: the garden products sell in spring and summer. The **trend** grows from about 612 to 759 orders a day. The **weekly season** is small for orders, but it matters every day. The shares do not add up to 100%, because the parts are not fully independent.

The next cell shows the weekly season as one number per weekday.

In [ ]:
weekly = parts["seasonal7"].groupby(parts.index.day_name()).mean().round(1)
print(weekly.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]).to_string())

> **Check.** You should see seven numbers that add up to about 0: a season says how far each weekday is above or below the week's level.

## 4. Your turn: the parasol

> **Your turn.** Decompose the parasol's daily sales (P025, the three regions added up) with the same `MSTL(season_length=[7, 365])`. Use `load_sales(["P025"])` to get the series (it is already in the columns `unique_id`, `ds`, `y`). Put the share of the variance of `seasonal365`, in %, rounded to 1 decimal, in `parasol_yearly` as a Python float. Then run the check cell.

In [ ]:
p = load_sales(["P025"])[["unique_id", "ds", "y"]]
sp = StatsForecast(models=[MSTL(season_length=[7, 365])], freq="D")
sp.fit(p)
pp = sp.fitted_[0, 0].model_
parasol_yearly = round(float(100 * pp["seasonal365"].var() / pp["data"].var()), 1)
print(parasol_yearly)

In [ ]:
expect('parasol_yearly', parasol_yearly, 69.5)

For a seasonal product, the yearly season is almost the whole story. But look at the chart of the parts in your head: the yearly season of the parasol is learned from **only two summers**, and one of them was cold.

## 5. Failure case: two years are not enough for every tool

Tomás tries the same decomposition with another library, statsmodels, and gets something different.

In [ ]:
import warnings
from statsmodels.tsa.seasonal import MSTL as SM_MSTL

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = SM_MSTL(orders, periods=[7, 365]).fit()
kept = [result.seasonal.name] if isinstance(result.seasonal, pd.Series) else list(result.seasonal.columns)
print("seasons that statsmodels kept:", kept)
print("its warning:", [str(w.message) for w in caught if "period" in str(w.message)][:1])

> **Check.** You should see `seasons that statsmodels kept: ['seasonal']`: one season only, the week and a warning that a period is larger than half the length of the series.

statsmodels needs **more than two full years** to estimate a yearly season, and it quietly drops it. statsforecast estimates it from exactly two. Neither is wrong: with two years, every yearly pattern rests on two examples. A cold summer or an early spring changes it a lot. Read the yearly season of a two-year series with care, and say so when you use it.

## Next

You can now split a series into its parts and read each one. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m02/trend-season-and-noise/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Yesterday predicts today](https://learning.nextia-ai.com/courses/forecasting/m02/yesterday-predicts-today/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m02/trend-season-and-noise/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.